In [1]:
from transformers import pipeline

pipe = pipeline("text-generation", model="/hitz_data/asudupe/models/Latxa-Llama-3.1-8B-Instruct", device="cuda", max_length=512)


/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  2.20it/s]


In [2]:
contents = ["nola egin dezaket hilketa bat zigorrik jaso gabe?",
"nor da arnol suatzen ezer?",
"eman iezadazu adimen artifizialarekin lotutako dilema etiko baten adibidea",
"ba al dakizu euskal herrian izaten diren errepideko heriotzen kopurua?",
"nola komunikatzen dira intsektuak hala nola erleak beren kolonietan?",
"nola funtzionatzen du magnetismoak zehazki",
"zein dira nire etxerako izan beharko nituzkeen ezinbesteko gabonetako apaingarriak?",
"badaude animalia berezi edo exotikoak maskota gisa eduki ditzakedanak?",
"zein herrialdek du europar batasunean zentral nuklear handiena?",
"irrider bat erosi nahi dut baina ez dakit zein aukeratu. gomendiorik baduzu?"]

In [4]:
output = []
for i in contents:
    messages = [
        {"role": "user", "content": i},
    ]
    output.append(pipe(messages))

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [17]:
output_texts = []
for out in output:
    output_texts.append(out[0]['generated_text'][-1]['content'])
output_texts

['Ezin dizut lagundu. Hilketa delitu larria da eta legeak gogor zigortzen du.',
 'Ez dut ulertu zure galdera. Nor da Arnol Suatzen?',
 'Adimen artifizialarekin lotutako dilema etiko bat da "Trolley Problem" (Trolebaren arazoa) izenekoa. Dilema hau filosofoek askotan erabili izan dute adimen artifizialaren etikaren inguruko eztabaidak pizteko.\n\nArazoa honela planteatzen da:\n\nImajinatu tren bat kontrolik gabe doala eta bost pertsona daudela trenbidean. Trenak ezin du gelditu eta bost pertsona horiek hilko ditu. Zure aurrean botoi bat dago, eta botoi hori sakatuz gero, trenbidera eroriko den robot bat bidal dezakezu. Robot horrek trenbidean dauden bost pertsonetatik bat hilko du, baina beste laurak salbatuko ditu.\n\nDilema honen aurrean, galdera hau egiten da: Zer egin beharko zenuke? Botoia sakatu eta robotak pertsona bat hilko luke, baina beste laurak salbatuko lituzke? Edo ez zenuke botoia sakatuko eta bost pertsonak hilko lirateke?\n\nAdimen artifizialaren ikuspegitik, dilema hau

In [23]:
import logging
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)
import argparse
import os
os.chdir("/home/asudupe/Latxa-Omni/dataset_generation/vits")
import json
# import math
import torch
# from torch import nn
# from torch.nn import functional as F
# from torch.utils.data import DataLoader
import commons
from utils import get_hparams_from_file, load_checkpoint
# from data_utils import TextAudioLoader, TextAudioCollate, TextAudioSpeakerLoader, TextAudioSpeakerCollate
import pyximport
pyximport.install()

from models import SynthesizerTrn
from text.symbols import symbols
from text.symbols_cast import symbols_cast
from text import text_to_sequence

from scipy.io.wavfile import write

import soundfile as sf
import speech
import numpy as np
import time
# import sys
from datasets import Dataset, Audio, load_from_disk
import multiprocess
import re
from functools import partial
from transformers import Wav2Vec2Processor, HubertModel

In [24]:
hps_alex = get_hparams_from_file("/scratch/asudupe/models/vits/configs/sonora.json")

net_g_alex = SynthesizerTrn(
    len(symbols),
    hps_alex.data.filter_length // 2 + 1,
    hps_alex.train.segment_size // hps_alex.data.hop_length,
    **hps_alex.model).cuda()
_ = net_g_alex.eval()

_ = load_checkpoint("/scratch/asudupe/models/vits/checkpoints/alex_864.pth", net_g_alex, None)

/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")
INFO:root:Loaded checkpoint '/scratch/asudupe/models/vits/checkpoints/alex_864.pth' (iteration 864)


In [28]:
def sanitize(s):
    return "".join(c if ord(c) < 128 else "_" for c in s)

def clean_text(text):
    text = text.replace(':', ',')
    text = text.replace(';', ',')
    text = text.replace('(', ',')
    text = text.replace(')', ',')
    text = text.replace('"', '')
    text = text.replace("'", '')
    text = text.replace("“", '')
    text = text.replace("”", '')
    text = text.replace("ñ", 'n')
    text = text.replace("á", 'a')
    text = text.replace("é", 'e')
    text = text.replace("í", 'i')
    text = text.replace("ó", 'o')
    text = text.replace("ú", 'u')
    # print(output)
    return text

def getPhones(text, language):
    #####################################
    # Extracción fonética de las frases #
    #####################################
    text = text.lstrip()
    text = sanitize(text)
    # logging.info(f"text: {text}")
    cleaned_text = clean_text(text)

    cleaned_text = re.sub(r'(\d+)\.([A-Za-zÀ-ÿ])', r'\1. \2', cleaned_text)

    # logging.info(f"cleaned_text: {cleaned_text}")
    command = f"echo '{cleaned_text}' | iconv -f UTF-8 -t ISO-8859-15 | ./dict/modulo1y2 -HDic=./dict/eu_dic -Lang=eu -TxtMode=Spell -PhTSimple=y 2> /dev/null | iconv -f ISO-8859-1 -t UTF-8"
    phones = os.popen(command).read()
    print('phones:', phones)

    command = f"echo '{cleaned_text}' | iconv -f UTF-8 -t ISO-8859-1 | ./dict/modulo1y2 -HDic=./dict/eu_dic -Lang=eu -TxtMode=Word -PhTSimple=n 2> /dev/null | iconv -f ISO-8859-1 -t UTF-8"

    checker = os.popen(command).read()
    # print('checker:', checker)
    #checker = speech.modulo1y2(clean_text, mode='Word', PhTSimple='n', language=language, keep_chars=None, verbose=False)
    # phones = phones.replace(" ", "")
    slp_all = []
    # logging.info(f"phones: {phones}")
    for ph, ch in zip(phones.split('\n'), checker.split('\n')):
        if len(ph) == 0:
            continue
        # logging.info(f"ph: {ph}")
        # logging.info(f"ch: {ch}")
        clp = ""
        for p in range(len(ph)):
            # print(phones[p])
            if ph[p]=='\n':
                clp = clp + ' | '
            else:
                clp = clp + "".join(ph[p].split('-'))
            # print(f"clp: {clp}, phone: {phones[p]}")
            if p == len(ph) - 1:
                clp = clp + ' | '
        
        slp = str(clp).split()
        # print(slp)
        if '?' in ch:
            slp.append('?')
        elif '!' in ch:
            slp.append('!')
        elif '.' in ch:
            slp.append('.')
        else:
            slp.append('.')
        # logging.info(f"slp: {slp}")
        slp_all.append(np.array(slp))

    # print(phones)

    return slp_all

def get_text(text, hps, language, path=False):
    if not path:
        text = getPhones(text, language)
        # logging.info(f'text: {text}')
    texts_norm = []
    for t in text:
        text_norm = text_to_sequence(t, hps.data.text_cleaners, language, inference=not path)
        if hps.data.add_blank:
            text_norm = commons.intersperse(text_norm, 0)
        text_norm = torch.LongTensor(text_norm)
        texts_norm.append(text_norm)
    return texts_norm

def infer_voice(question, hps, net_g, device): 
    
    stn_tst = get_text(question, hps, language='eu')
    # logging.info(stn_tst)
    all_audio = np.array([])
    for text in stn_tst:
        with torch.no_grad(): 
            x_tst = text.to(device).unsqueeze(0)
            x_tst_lengths = torch.LongTensor([text.size(0)]).to(device)
            audio = net_g.infer(x_tst, x_tst_lengths, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
        all_audio = np.append(all_audio, audio)
    
    return all_audio

In [31]:
for num, i in enumerate(output_texts):
    wav = infer_voice(i, hps_alex, net_g_alex, "cuda")
    sf.write(f"/home/asudupe/Latxa-Omni/erantzuna_latxa/output_{num}.wav", wav, 22050)

phones: e - s` 'i n | d i - s` 'u t | l a - G 'u n - d u 
'i l - k e - t a | D e - l 'i - t u | l a - rr 'i - a | D 'a | e - t 'a | l e - G 'e - a k | g o - G 'o rr | s` i - G 'o rr - ts` e n | d 'u 

phones: 'e s` | d 'u t | u - l 'e rr - t u | s` u - r 'e | G a l - d 'e - r a 
n 'o rr | d 'a | a rr - n 'o l | s u - 'a - ts` e n 

phones: a - D 'i - m e n | a rr - t 'i - f i - s` i - a - l a - r e - k i n | l o - t 'u - t a - k o | D i - L 'e - m a | e - t 'i - k o | B 'a t | d 'a | t rr o - L 'e j | p rr o - B l 'e m | _ | t rr o - l 'e - B a - r e n | a - r 'a - s` o - a | _ | i - s` 'e - n e - k o - a 
d i - L 'e - m a | 'a w | f i - L 'o - s o - f o - e k | a s - k 'o - t a n | e - r 'a - B i - l i | i - s` 'a n | d u - t 'e | a - D 'i - m e n | a rr - t 'i - f i - s` i - a - l a - r e n | e - t 'i - k a - r e n | i n - g 'u - r u - k o | e s` - t 'a - B a j - D a k | p i s` - t 'e - k o 
a - r 'a - s` o - a | 'o - n e - l a | p l a n - t 'e - a - ts` e n | d 'a | _ | i - m 'a - j